# Exploracao — fluxo assistencial do SUS como grafo

Le as tabelas tidy de `data/processed/` (gere com `python src/extracao.py --amostra`)
e mostra o fluxo de deslocamento inter-regional que vira o grafo: taxa de
deslocamento, indice de retencao regional (IRR) por complexidade e saldo migratorio
(invasao menos evasao). Fecha consultando o grafo ja carregado no Neo4j.

## 1. Carrega as tabelas de fluxo

In [ ]:
import pandas as pd
from pathlib import Path

PROC = Path("..") / "data" / "processed"
fluxo = pd.read_parquet(PROC / "fluxo.parquet")
regioes = pd.read_parquet(PROC / "regioes.parquet")
municipios = pd.read_parquet(PROC / "municipios.parquet")
print("arestas de fluxo:", len(fluxo))
fluxo.head()

## 2. Taxa de deslocamento e IRR por complexidade\n\nDeslocamento inter-regional = residente atendido fora da propria regiao. A retencao cai com a complexidade: a atencao basica e local, a alta se concentra em polos.

In [ ]:
tx = fluxo.loc[fluxo.deslocou, "volume"].sum() / fluxo["volume"].sum()
print(f"deslocamento global (ponderado por evento): {tx:.1%}\n")

for c in ["basica", "media", "alta"]:
    d = fluxo[fluxo.complexidade == c]
    irr = d.loc[~d.deslocou, "volume"].sum() / d["volume"].sum()
    print(f"IRR {c:>7}: {irr:.2f}   (taxa de deslocamento {1-irr:.0%})")

## 3. Saldo migratorio: quem e polo, quem e satelite\n\nInvasao = atendimentos que a regiao presta a residentes de fora. Evasao = atendimentos que seus residentes buscam fora. Saldo positivo indica polo.

In [ ]:
desloc = fluxo[fluxo.deslocou]
evasao = desloc.groupby("reg_res_nome")["volume"].sum().rename("evasao")
invasao = desloc.groupby("reg_aten_nome")["volume"].sum().rename("invasao")
saldo = pd.concat([invasao, evasao], axis=1).fillna(0)
saldo["saldo"] = saldo["invasao"] - saldo["evasao"]
saldo.sort_values("saldo", ascending=False).head(10)

## 4. Ranking de evasao (regioes que mais dependem de fora)

In [ ]:
evasao.sort_values(ascending=False).head(10).to_frame()

## 5. O mesmo, ja no grafo (Neo4j)

Com `docker compose up -d` e `python src/grafo.py` feitos, a mesma pergunta vira
uma consulta Cypher. E o que o Text2Cypher automatiza a partir do portugues.

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
load_dotenv("../.env")

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "neo4j")),
)
cypher = """
MATCH (o:RegiaoSaude)-[f:FLUXO]->(:RegiaoSaude)
WHERE f.deslocou = true
RETURN o.nome AS regiao, sum(f.volume) AS evasao
ORDER BY evasao DESC LIMIT 10
"""
with driver.session() as s:
    print(pd.DataFrame([dict(r) for r in s.run(cypher)]))
driver.close()